
# Roxy notebook example: Physicochemical autocorrelation descriptors

This notebook is a **reference implementation example** for the **physicochemical autocorrelation descriptor family** in Roxy.

Autocorrelation descriptors quantify how residue-level physicochemical values are related across a sequence at different **lags**. They help capture sequence-order information without requiring structure or embeddings.

## Covered outputs

This notebook implements three classical autocorrelation families over physicochemical scales:

- **Moreau-Broto autocorrelation**
- **Moran autocorrelation**
- **Geary autocorrelation**

Using several residue-level scales:

- hydrophobicity
- polarity
- flexibility
- volume
- charge proxy

For each selected lag, the notebook computes autocorrelation descriptors and demonstrates how they can be packaged into a class-style implementation for Roxy.


In [2]:

import numpy as np
import pandas as pd


## Demo dataset

In [3]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "auto_1",
            "auto_2",
            "auto_3",
            "auto_4",
            "auto_5",
            "auto_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,auto_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,auto_2,GGGGGGGGGGGGGGG,B
2,auto_3,KRRKRRKRRKRRDDDDEE,A
3,auto_4,ACDEFGHIKLMNPQRSTVWY,B
4,auto_5,PPPPGSSSSSTTTTNNQQQ,A
5,auto_6,MSTNPKPQRITLKDGNKVELV,B


## Constants and residue-level physicochemical scales

In [4]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

SCALES = {
    "hydrophobicity": {
        "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
        "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
        "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
        "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
    },
    "polarity": {
        "A": 8.1, "C": 5.5, "D": 13.0, "E": 12.3, "F": 5.2,
        "G": 9.0, "H": 10.4, "I": 5.2, "K": 11.3, "L": 4.9,
        "M": 5.7, "N": 11.6, "P": 8.0, "Q": 10.5, "R": 10.5,
        "S": 9.2, "T": 8.6, "V": 5.9, "W": 5.4, "Y": 6.2,
    },
    "flexibility": {
        "A": 0.357, "C": 0.346, "D": 0.511, "E": 0.497, "F": 0.314,
        "G": 0.544, "H": 0.323, "I": 0.462, "K": 0.466, "L": 0.365,
        "M": 0.295, "N": 0.463, "P": 0.509, "Q": 0.493, "R": 0.529,
        "S": 0.507, "T": 0.444, "V": 0.386, "W": 0.305, "Y": 0.420,
    },
    "volume": {
        "A": 88.6, "C": 108.5, "D": 111.1, "E": 138.4, "F": 189.9,
        "G": 60.1, "H": 153.2, "I": 166.7, "K": 168.6, "L": 166.7,
        "M": 162.9, "N": 114.1, "P": 112.7, "Q": 143.8, "R": 173.4,
        "S": 89.0, "T": 116.1, "V": 140.0, "W": 227.8, "Y": 193.6,
    },
    "charge_proxy": {
        "A": 0.0, "C": 0.0, "D": -1.0, "E": -1.0, "F": 0.0,
        "G": 0.0, "H": 0.5, "I": 0.0, "K": 1.0, "L": 0.0,
        "M": 0.0, "N": 0.0, "P": 0.0, "Q": 0.0, "R": 1.0,
        "S": 0.0, "T": 0.0, "V": 0.0, "W": 0.0, "Y": 0.0,
    },
}


## Helper functions

In [5]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def scale_values(seq: str, scale_name: str) -> np.ndarray:
    seq = clean_sequence(seq)
    scale = SCALES[scale_name]
    values = [scale[aa] for aa in seq if aa in scale]
    return np.array(values, dtype=float)


def zscore(values: np.ndarray) -> np.ndarray:
    if len(values) == 0:
        return values
    std = np.std(values, ddof=0)
    if std == 0:
        return np.zeros_like(values, dtype=float)
    return (values - np.mean(values)) / std


def moreau_broto(values: np.ndarray, lag: int) -> float:
    n = len(values)
    if n <= lag or lag < 1:
        return np.nan
    return float(np.sum(values[:-lag] * values[lag:]) / (n - lag))


def moran(values: np.ndarray, lag: int) -> float:
    n = len(values)
    if n <= lag or lag < 1:
        return np.nan
    mean_val = np.mean(values)
    denominator = np.mean((values - mean_val) ** 2)
    if denominator == 0:
        return np.nan
    numerator = np.sum((values[:-lag] - mean_val) * (values[lag:] - mean_val)) / (n - lag)
    return float(numerator / denominator)


def geary(values: np.ndarray, lag: int) -> float:
    n = len(values)
    if n <= lag or lag < 1:
        return np.nan
    mean_val = np.mean(values)
    denominator = np.mean((values - mean_val) ** 2)
    if denominator == 0:
        return np.nan
    numerator = np.sum((values[:-lag] - values[lag:]) ** 2) / (2 * (n - lag))
    return float(numerator / denominator)


## Core descriptor function

In [6]:

def autocorrelation_descriptors(
    seq: str,
    scales=("hydrophobicity", "polarity", "flexibility", "volume", "charge_proxy"),
    lags=(1, 2, 3, 4, 5),
) -> dict:
    seq = clean_sequence(seq)

    out = {
        "auto_length": len(seq),
        "auto_valid_residue_count": len(seq),
    }

    for scale_name in scales:
        values = scale_values(seq, scale_name)

        for lag in lags:
            out[f"auto_mb_{scale_name}_lag{lag}"] = moreau_broto(values, lag)
            out[f"auto_moran_{scale_name}_lag{lag}"] = moran(values, lag)
            out[f"auto_geary_{scale_name}_lag{lag}"] = geary(values, lag)

    return out


## Functional usage on one sequence

In [7]:

example = autocorrelation_descriptors(df_demo.loc[0, "sequence"], lags=(1, 2, 3))
list(example.items())[:18]


[('auto_length', 24),
 ('auto_valid_residue_count', 24),
 ('auto_mb_hydrophobicity_lag1', 2.5447826086956518),
 ('auto_moran_hydrophobicity_lag1', 0.2332481093779379),
 ('auto_geary_hydrophobicity_lag1', 0.7403545701443606),
 ('auto_mb_hydrophobicity_lag2', -0.7536363636363639),
 ('auto_moran_hydrophobicity_lag2', -0.17831898035390062),
 ('auto_geary_hydrophobicity_lag2', 1.073545221265357),
 ('auto_mb_hydrophobicity_lag3', 1.8585714285714279),
 ('auto_moran_hydrophobicity_lag3', 0.121713250053193),
 ('auto_geary_hydrophobicity_lag3', 0.7968916495124616),
 ('auto_mb_polarity_lag1', 53.35434782608695),
 ('auto_moran_polarity_lag1', 0.13031618709541473),
 ('auto_geary_polarity_lag1', 0.8560279885411344),
 ('auto_mb_polarity_lag2', 49.382727272727266),
 ('auto_moran_polarity_lag2', -0.18923501798123885),
 ('auto_geary_polarity_lag2', 1.0978121773889926),
 ('auto_mb_polarity_lag3', 52.37523809523809)]

## Apply autocorrelation descriptors to the full dataset

In [8]:

df_auto = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(lambda x: autocorrelation_descriptors(x, lags=(1, 2, 3, 4, 5))).apply(pd.Series),
    ],
    axis=1,
)

df_auto.head()


,sequence_id,sequence,label,auto_length,auto_valid_residue_count,auto_mb_hydrophobicity_lag1,auto_moran_hydrophobicity_lag1,auto_geary_hydrophobicity_lag1,auto_mb_hydrophobicity_lag2,auto_moran_hydrophobicity_lag2,...,auto_geary_charge_proxy_lag2,auto_mb_charge_proxy_lag3,auto_moran_charge_proxy_lag3,auto_geary_charge_proxy_lag3,auto_mb_charge_proxy_lag4,auto_moran_charge_proxy_lag4,auto_geary_charge_proxy_lag4,auto_mb_charge_proxy_lag5,auto_moran_charge_proxy_lag5,auto_geary_charge_proxy_lag5
0,auto_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,2.544783,0.233248,0.740355,-0.753636,-0.178319,...,0.818182,0.000000,-0.085714,0.857143,0.050000,0.260000,0.540000,0.052632,0.263158,0.568421
1,auto_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.160000,1.000000,0.000000,0.160000,1.000000,...,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN
2,auto_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,16.520588,0.483957,0.529412,16.506250,0.274148,...,0.281250,0.600000,0.500000,0.450000,0.428571,0.285714,0.642857,0.230769,0.038462,0.865385
3,auto_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,-1.172632,-0.171179,1.205492,-0.620556,-0.114581,...,0.983284,-0.029412,-0.149748,1.179941,-0.031250,-0.166667,1.106195,-0.066667,-0.335300,1.337266
4,auto_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,3.593333,0.792265,0.192536,3.119412,0.580801,...,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN


## Inspect autocorrelation descriptor columns

In [9]:

auto_cols = [c for c in df_auto.columns if c.startswith("auto_") and c not in {"auto_length", "auto_valid_residue_count"}]
len(auto_cols), auto_cols[:15]


(75,
 ['auto_mb_hydrophobicity_lag1',
  'auto_moran_hydrophobicity_lag1',
  'auto_geary_hydrophobicity_lag1',
  'auto_mb_hydrophobicity_lag2',
  'auto_moran_hydrophobicity_lag2',
  'auto_geary_hydrophobicity_lag2',
  'auto_mb_hydrophobicity_lag3',
  'auto_moran_hydrophobicity_lag3',
  'auto_geary_hydrophobicity_lag3',
  'auto_mb_hydrophobicity_lag4',
  'auto_moran_hydrophobicity_lag4',
  'auto_geary_hydrophobicity_lag4',
  'auto_mb_hydrophobicity_lag5',
  'auto_moran_hydrophobicity_lag5',
  'auto_geary_hydrophobicity_lag5'])

In [10]:

df_auto[
    [
        "sequence_id",
        "auto_mb_hydrophobicity_lag1",
        "auto_moran_hydrophobicity_lag1",
        "auto_geary_hydrophobicity_lag1",
        "auto_mb_charge_proxy_lag2",
        "auto_moran_charge_proxy_lag2",
        "auto_geary_charge_proxy_lag2",
    ]
]


,sequence_id,auto_mb_hydrophobicity_lag1,auto_moran_hydrophobicity_lag1,auto_geary_hydrophobicity_lag1,auto_mb_charge_proxy_lag2,auto_moran_charge_proxy_lag2,auto_geary_charge_proxy_lag2
0,auto_1,2.544783,0.233248,0.740355,0.000000,-0.072727,0.818182
1,auto_2,0.160000,1.000000,0.000000,0.000000,NaN,NaN
2,auto_3,16.520588,0.483957,0.529412,0.750000,0.687500,0.281250
3,auto_4,-1.172632,-0.171179,1.205492,0.027778,0.127499,0.983284
4,auto_5,3.593333,0.792265,0.192536,0.000000,NaN,NaN
5,auto_6,0.086000,-0.089975,1.057968,-0.052632,-0.229940,1.331752


## Dataset-level summary

In [11]:

auto_summary = (
    df_auto[auto_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

auto_summary.head(15)


,descriptor,mean_value
0,auto_mb_volume_lag1,16807.444271
1,auto_mb_volume_lag5,16686.197097
2,auto_mb_volume_lag4,16556.921510
3,auto_mb_volume_lag2,16492.169720
4,auto_mb_volume_lag3,16325.708018
5,auto_mb_polarity_lag3,84.528526
6,auto_mb_polarity_lag2,84.077287
7,auto_mb_polarity_lag1,83.813381
8,auto_mb_polarity_lag5,83.014727
9,auto_mb_polarity_lag4,82.461950


## Sanity checks

In [12]:

assert "auto_mb_hydrophobicity_lag1" in df_auto.columns
assert "auto_moran_polarity_lag2" in df_auto.columns
assert "auto_geary_flexibility_lag3" in df_auto.columns
assert "auto_mb_volume_lag4" in df_auto.columns
assert "auto_moran_charge_proxy_lag5" in df_auto.columns
assert df_auto["auto_length"].min() > 0

print(f"Number of autocorrelation descriptor columns: {len(auto_cols)}")
print("Autocorrelation descriptor checks passed.")


Number of autocorrelation descriptor columns: 75
Autocorrelation descriptor checks passed.


## Class-style implementation closer to the real package

In [13]:

class PhysicochemicalAutocorrelationDescriptors:
    """Example class-style autocorrelation implementation for later migration into Roxy."""

    def __init__(
        self,
        scales=("hydrophobicity", "polarity", "flexibility", "volume", "charge_proxy"),
        lags=(1, 2, 3, 4, 5),
    ):
        self.scales = tuple(scales)
        self.lags = tuple(lags)

    def transform_sequence(self, seq: str) -> dict:
        return autocorrelation_descriptors(
            seq,
            scales=self.scales,
            lags=self.lags,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


auto_transformer = PhysicochemicalAutocorrelationDescriptors(
    scales=("hydrophobicity", "polarity", "flexibility", "volume", "charge_proxy"),
    lags=(1, 2, 3, 4, 5),
)

auto_matrix = auto_transformer.transform(df_demo["sequence"].tolist())
auto_matrix.head()


,auto_length,auto_valid_residue_count,auto_mb_hydrophobicity_lag1,auto_moran_hydrophobicity_lag1,auto_geary_hydrophobicity_lag1,auto_mb_hydrophobicity_lag2,auto_moran_hydrophobicity_lag2,auto_geary_hydrophobicity_lag2,auto_mb_hydrophobicity_lag3,auto_moran_hydrophobicity_lag3,...,auto_geary_charge_proxy_lag2,auto_mb_charge_proxy_lag3,auto_moran_charge_proxy_lag3,auto_geary_charge_proxy_lag3,auto_mb_charge_proxy_lag4,auto_moran_charge_proxy_lag4,auto_geary_charge_proxy_lag4,auto_mb_charge_proxy_lag5,auto_moran_charge_proxy_lag5,auto_geary_charge_proxy_lag5
0,24,24,2.544783,0.233248,0.740355,-0.753636,-0.178319,1.073545,1.858571,0.121713,...,0.818182,0.000000,-0.085714,0.857143,0.050000,0.260000,0.540000,0.052632,0.263158,0.568421
1,15,15,0.160000,1.000000,0.000000,0.160000,1.000000,0.000000,0.160000,1.000000,...,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN
2,18,18,16.520588,0.483957,0.529412,16.506250,0.274148,0.722301,16.602000,0.609091,...,0.281250,0.600000,0.500000,0.450000,0.428571,0.285714,0.642857,0.230769,0.038462,0.865385
3,20,20,-1.172632,-0.171179,1.205492,-0.620556,-0.114581,1.176506,1.587059,0.139397,...,0.983284,-0.029412,-0.149748,1.179941,-0.031250,-0.166667,1.106195,-0.066667,-0.335300,1.337266
4,19,19,3.593333,0.792265,0.192536,3.119412,0.580801,0.387014,2.586250,0.342904,...,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN


## Merge transformer output back to the dataset

In [14]:

df_auto_class = pd.concat([df_demo, auto_matrix], axis=1)
df_auto_class.head()


,sequence_id,sequence,label,auto_length,auto_valid_residue_count,auto_mb_hydrophobicity_lag1,auto_moran_hydrophobicity_lag1,auto_geary_hydrophobicity_lag1,auto_mb_hydrophobicity_lag2,auto_moran_hydrophobicity_lag2,...,auto_geary_charge_proxy_lag2,auto_mb_charge_proxy_lag3,auto_moran_charge_proxy_lag3,auto_geary_charge_proxy_lag3,auto_mb_charge_proxy_lag4,auto_moran_charge_proxy_lag4,auto_geary_charge_proxy_lag4,auto_mb_charge_proxy_lag5,auto_moran_charge_proxy_lag5,auto_geary_charge_proxy_lag5
0,auto_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,2.544783,0.233248,0.740355,-0.753636,-0.178319,...,0.818182,0.000000,-0.085714,0.857143,0.050000,0.260000,0.540000,0.052632,0.263158,0.568421
1,auto_2,GGGGGGGGGGGGGGG,B,15,15,0.160000,1.000000,0.000000,0.160000,1.000000,...,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN
2,auto_3,KRRKRRKRRKRRDDDDEE,A,18,18,16.520588,0.483957,0.529412,16.506250,0.274148,...,0.281250,0.600000,0.500000,0.450000,0.428571,0.285714,0.642857,0.230769,0.038462,0.865385
3,auto_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,-1.172632,-0.171179,1.205492,-0.620556,-0.114581,...,0.983284,-0.029412,-0.149748,1.179941,-0.031250,-0.166667,1.106195,-0.066667,-0.335300,1.337266
4,auto_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,3.593333,0.792265,0.192536,3.119412,0.580801,...,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move physicochemical scales into `roxy/core/constants.py`
- move helper logic into `roxy/sequence/autocorrelation.py`
- expose a class such as `PhysicochemicalAutocorrelationDescriptors`
- allow configurable:
  - selected scales
  - selected lags
  - selected autocorrelation families (Moreau-Broto, Moran, Geary)
- add tests for:
  - empty sequences
  - very short sequences where lag > length
  - constant-valued sequences under one scale
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [15]:
# df_auto.to_csv("demo_autocorrelation_descriptors.csv", index=False)
